# 300k + VAE Fidelity Benchmark

## Install requirements

In [ ]:
%pip install -q "numpy<2" pandas scikit-learn matplotlib cloudpickle interpret tensorflow==2.15.1 tensorflow-lattice gaminet

## Imports and paths

In [ ]:
import os
import pickle
import sys
import warnings
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import cloudpickle
import numpy as np
import pandas as pd
from IPython.display import Image, display
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

ROOT = Path.cwd()
MODEL_DIR = ROOT / "model"
CSV_DIR = ROOT / "processed_csv"
PLOT_DIR = ROOT / "plots_300k_vae"
PREDICT_BATCH_SIZE = 50000

TEACHERS = ["random_forest", "deep_neural_net"]
SURROGATES = ["logistic_regression", "ebm", "gaminet"]

## Helper functions

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }


def batched_predict(model, X, batch_size=PREDICT_BATCH_SIZE):
    preds = []
    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))
        batch = X.iloc[start:stop] if hasattr(X, "iloc") else X[start:stop]
        preds.append(np.asarray(model.predict(batch)).astype(int))
    return np.concatenate(preds, axis=0)


def gaminet_predict_labels(model, X, batch_size=8192):
    preds = []
    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))
        batch_pred = np.asarray(model.predict(X[start:stop]), dtype=np.float32).reshape(-1)
        if np.nanmin(batch_pred) < 0.0 or np.nanmax(batch_pred) > 1.0:
            batch_pred = 1.0 / (1.0 + np.exp(-batch_pred))
        preds.append((batch_pred >= 0.5).astype(np.int32))
    return np.concatenate(preds, axis=0)


def load_cloudpickle(path):
    if "numpy._core.numeric" not in sys.modules:
        import numpy.core.numeric as numpy_core_numeric
        sys.modules["numpy._core.numeric"] = numpy_core_numeric
    with path.open("rb") as handle:
        return cloudpickle.load(handle)


def _import_optional(module_name):
    try:
        return __import__(module_name, fromlist=["*"])
    except Exception:
        return None


def ensure_legacy_keras_module(module_name):
    if module_name in sys.modules:
        return True
    if not module_name.startswith("keras."):
        return False

    if module_name == "keras.mixed_precision.policy":
        import types

        policy_module = types.ModuleType("keras.mixed_precision.policy")
        source_modules = []
        for candidate in ("keras.mixed_precision", "tensorflow.keras.mixed_precision"):
            source = _import_optional(candidate)
            if source is not None:
                source_modules.append(source)

        if not source_modules:
            return False

        for source in source_modules:
            for attr in ("Policy", "global_policy", "set_global_policy", "LossScaleOptimizer"):
                if hasattr(source, attr) and not hasattr(policy_module, attr):
                    setattr(policy_module, attr, getattr(source, attr))

        sys.modules[module_name] = policy_module
        return True

    tf_equivalent = module_name.replace("keras.", "tensorflow.keras.", 1)
    parent = ".".join(module_name.split(".")[:-1])
    tf_parent = ".".join(tf_equivalent.split(".")[:-1])

    candidates = [
        tf_equivalent,
        parent if parent else None,
        tf_parent if tf_parent else None,
        "keras.layers" if module_name.startswith("keras.layers.") else None,
        "tensorflow.keras.layers" if module_name.startswith("keras.layers.") else None,
        "keras.models" if module_name.startswith("keras.engine.") else None,
        "tensorflow.keras.models" if module_name.startswith("keras.engine.") else None,
        "keras",
        "tensorflow.keras",
    ]

    for candidate in candidates:
        if not candidate:
            continue
        module = _import_optional(candidate)
        if module is not None:
            sys.modules[module_name] = module
            return True

    return False


def ensure_legacy_keras_attribute(module_name, attribute_name):
    target_module = _import_optional(module_name)
    if target_module is None and module_name.startswith("keras."):
        if ensure_legacy_keras_module(module_name):
            target_module = _import_optional(module_name)
    if target_module is None:
        return False
    if hasattr(target_module, attribute_name):
        return True

    tf_equivalent = module_name.replace("keras.", "tensorflow.keras.", 1) if module_name.startswith("keras.") else None
    candidate_modules = [
        tf_equivalent,
        "keras",
        "keras.utils",
        "keras.layers",
        "keras.models",
        "keras.losses",
        "keras.metrics",
        "keras.optimizers",
        "keras.initializers",
        "keras.regularizers",
        "keras.constraints",
        "keras.activations",
        "keras.backend",
        "keras.saving",
        "tensorflow.keras",
        "tensorflow.keras.utils",
        "tensorflow.keras.layers",
        "tensorflow.keras.models",
        "tensorflow.keras.losses",
        "tensorflow.keras.metrics",
        "tensorflow.keras.optimizers",
        "tensorflow.keras.initializers",
        "tensorflow.keras.regularizers",
        "tensorflow.keras.constraints",
        "tensorflow.keras.activations",
        "tensorflow.keras.backend",
    ]

    for candidate in candidate_modules:
        if not candidate:
            continue
        source = _import_optional(candidate)
        if source is not None and hasattr(source, attribute_name):
            setattr(target_module, attribute_name, getattr(source, attribute_name))
            return True

    for loaded_name, loaded_module in list(sys.modules.items()):
        if not loaded_name or loaded_module is None:
            continue
        if (loaded_name.startswith("keras") or loaded_name.startswith("tensorflow.keras")) and hasattr(loaded_module, attribute_name):
            setattr(target_module, attribute_name, getattr(loaded_module, attribute_name))
            return True

    keras_src = _import_optional("keras.src")
    if keras_src is not None and hasattr(keras_src, "__path__"):
        import pkgutil

        allowed_prefixes = ("keras.src.utils.", "keras.src.saving.", "keras.src.layers.", "keras.src.models.", "keras.src.engine.")
        for module_info in pkgutil.walk_packages(keras_src.__path__, prefix="keras.src."):
            if not module_info.name.startswith(allowed_prefixes):
                continue
            source = _import_optional(module_info.name)
            if source is not None and hasattr(source, attribute_name):
                setattr(target_module, attribute_name, getattr(source, attribute_name))
                return True

    return False


def parse_legacy_keras_attribute_error(exc):
    message = str(exc)
    attr_marker = "Can't get attribute '"
    module_marker = "on <module '"
    if attr_marker not in message or module_marker not in message:
        return None, None
    attribute_name = message.split(attr_marker, 1)[1].split("'", 1)[0]
    module_name = message.split(module_marker, 1)[1].split("'", 1)[0]
    return attribute_name, module_name


def ensure_legacy_keras_module_aliases():
    ensure_legacy_keras_module("keras.mixed_precision.policy")
    ensure_legacy_keras_attribute("keras", "InputSpec")


def ensure_gaminet_runtime():
    import tensorflow as tf
    from gaminet import GAMINet
    try:
        tf.config.set_visible_devices([], "GPU")
    except Exception:
        pass
    try:
        tf.config.experimental.set_visible_devices([], "GPU")
    except Exception:
        pass
    return tf, GAMINet


def load_gaminet_bundle(model_path):
    _, GAMINet = ensure_gaminet_runtime()
    ensure_legacy_keras_module_aliases()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with model_path.open("rb") as handle:
            attempted_missing_modules = set()
            attempted_missing_attrs = set()
            while True:
                try:
                    model_dict = pickle.load(handle)
                    break
                except ModuleNotFoundError as exc:
                    missing_module = getattr(exc, "name", "") or ""
                    if (
                        not missing_module
                        or missing_module in attempted_missing_modules
                        or not ensure_legacy_keras_module(missing_module)
                    ):
                        raise
                    attempted_missing_modules.add(missing_module)
                    handle.seek(0)
                except AttributeError as exc:
                    missing_attr, missing_attr_module = parse_legacy_keras_attribute_error(exc)
                    missing_key = f"{missing_attr_module}:{missing_attr}"
                    if (
                        not missing_attr
                        or not missing_attr_module
                        or missing_key in attempted_missing_attrs
                        or not ensure_legacy_keras_attribute(missing_attr_module, missing_attr)
                    ):
                        raise
                    attempted_missing_attrs.add(missing_key)
                    handle.seek(0)
    model = GAMINet(
        meta_info=model_dict["meta_info"],
        subnet_arch=model_dict["subnet_arch"],
        interact_arch=model_dict["interact_arch"],
        lr_bp=model_dict["lr_bp"],
        batch_size=model_dict["batch_size"],
        task_type=model_dict["task_type"],
        activation_func=model_dict["activation_func"],
        tuning_epochs=model_dict["tuning_epochs"],
        main_effect_epochs=model_dict["main_effect_epochs"],
        interaction_epochs=model_dict["interaction_epochs"],
        early_stop_thres=model_dict["early_stop_thres"],
        heredity=model_dict["heredity"],
        reg_clarity=model_dict["reg_clarity"],
        loss_threshold=model_dict["loss_threshold"],
        mono_increasing_list=model_dict["mono_increasing_list"],
        mono_decreasing_list=model_dict["mono_decreasing_list"],
        lattice_size=model_dict["lattice_size"],
        verbose=False,
        val_ratio=model_dict["val_ratio"],
        random_state=model_dict["random_state"],
        interact_num=model_dict["interact_num"],
    )
    model.load(folder=str(model_path.parent) + "/", name=model_path.stem)
    return model, model_dict


def scale_with_saved_meta_info(X, meta_info):
    feature_names = [name for name, payload in meta_info.items() if payload.get("type") != "target"]
    X_np = X[feature_names].astype(np.float32, copy=False).to_numpy(dtype=np.float32, copy=True)
    X_scaled = np.zeros_like(X_np, dtype=np.float32)
    for idx, feature_name in enumerate(feature_names):
        scaler = meta_info[feature_name]["scaler"]
        X_scaled[:, [idx]] = scaler.transform(X_np[:, [idx]]).astype(np.float32)
    return X_scaled, feature_names


def error_report(teacher_pred, surrogate_pred, y_true):
    masks = {
        "overall_error": teacher_pred != y_true,
        "false_positive": (teacher_pred == 1) & (y_true == 0),
        "false_negative": (teacher_pred == 0) & (y_true == 1),
    }
    out = {}
    for name, mask in masks.items():
        out[f"{name}_count"] = int(mask.sum())
        out[f"{name}_fidelity"] = float(np.mean(surrogate_pred[mask] == teacher_pred[mask])) if mask.any() else np.nan
        out[f"{name}_truth_match"] = float(np.mean(surrogate_pred[mask] == y_true[mask])) if mask.any() else np.nan
    return out

## Load test data and teacher predictions

In [ ]:
processed = pd.read_csv(CSV_DIR / "processed_dataset_with_split.csv")
feature_cols = [col for col in processed.columns if col not in {"label", "split"}]
test_df = processed[processed["split"] == "test"].copy()
X_test = test_df[feature_cols].astype(np.float32)
y_test = test_df["label"].to_numpy(dtype=int, copy=True)
teacher_predictions = {
    "random_forest": np.load(MODEL_DIR / "teacher_real_test_predictions_random_forest.npy").astype(int),
    "deep_neural_net": np.load(MODEL_DIR / "teacher_real_test_predictions_deep_neural_net.npy").astype(int),
}
pd.DataFrame({"split": ["test"], "rows": [len(test_df)], "features": [len(feature_cols)]})

## Load surrogate models

In [ ]:
models = {
    "random_forest": {
        "logistic_regression": load_cloudpickle(MODEL_DIR / "logreg_mixed_300k_plus_vae_random_forest.pkl"),
        "ebm": load_cloudpickle(MODEL_DIR / "ebm_random_forest.pkl"),
    },
    "deep_neural_net": {
        "logistic_regression": load_cloudpickle(MODEL_DIR / "logreg_mixed_300k_plus_vae_deep_neural_net.pkl"),
        "ebm": load_cloudpickle(MODEL_DIR / "ebm_deep_neural_net.pkl"),
    },
}
gaminet_bundles = {
    "random_forest": load_gaminet_bundle(MODEL_DIR / "gaminet_random_forest.pickle"),
    "deep_neural_net": load_gaminet_bundle(MODEL_DIR / "gaminet_deep_neural_net.pickle"),
}
models["random_forest"]["gaminet"] = gaminet_bundles["random_forest"][0]
models["deep_neural_net"]["gaminet"] = gaminet_bundles["deep_neural_net"][0]
list(models.keys())

## Run surrogate inference

In [ ]:
predictions = {teacher: {} for teacher in TEACHERS}
for teacher in TEACHERS:
    predictions[teacher]["logistic_regression"] = batched_predict(models[teacher]["logistic_regression"], X_test)
    predictions[teacher]["ebm"] = batched_predict(models[teacher]["ebm"], X_test)
    gaminet_model, gaminet_payload = gaminet_bundles[teacher]
    X_gaminet, gaminet_feature_names = scale_with_saved_meta_info(X_test, gaminet_payload["meta_info"])
    predictions[teacher]["gaminet"] = gaminet_predict_labels(gaminet_model, X_gaminet)
pd.DataFrame({"teacher": TEACHERS, "surrogates": [list(predictions[t].keys()) for t in TEACHERS]})

## Overall fidelity and F1-fidelity

In [ ]:
rows = []
for teacher in TEACHERS:
    teacher_pred = teacher_predictions[teacher]
    for surrogate in SURROGATES:
        pred = predictions[teacher][surrogate]
        fidelity = compute_metrics(teacher_pred, pred)
        truth = compute_metrics(y_test, pred)
        rows.append({
            "teacher": teacher,
            "surrogate": surrogate,
            "test_accuracy_vs_truth": truth["accuracy"],
            "overall_fidelity_accuracy": fidelity["accuracy"],
            "f1_fidelity": fidelity["f1"],
            "fidelity_precision": fidelity["precision"],
            "fidelity_recall": fidelity["recall"],
        })
overall_fidelity_df = pd.DataFrame(rows)
overall_fidelity_df

## Error fidelity on teacher mistakes

In [ ]:
rows = []
for teacher in TEACHERS:
    teacher_pred = teacher_predictions[teacher]
    for surrogate in SURROGATES:
        pred = predictions[teacher][surrogate]
        report = error_report(teacher_pred, pred, y_test)
        rows.append({
            "teacher": teacher,
            "surrogate": surrogate,
            "teacher_error_count": report["overall_error_count"],
            "error_fidelity": report["overall_error_fidelity"],
            "matches_truth_on_teacher_errors": report["overall_error_truth_match"],
        })
error_fidelity_df = pd.DataFrame(rows)
error_fidelity_df

## FP and FN error fidelity

In [ ]:
rows = []
for teacher in TEACHERS:
    teacher_pred = teacher_predictions[teacher]
    for surrogate in SURROGATES:
        pred = predictions[teacher][surrogate]
        report = error_report(teacher_pred, pred, y_test)
        rows.append({
            "teacher": teacher,
            "surrogate": surrogate,
            "fp_count": report["false_positive_count"],
            "fp_fidelity": report["false_positive_fidelity"],
            "fn_count": report["false_negative_count"],
            "fn_fidelity": report["false_negative_fidelity"],
        })
detailed_error_fidelity_df = pd.DataFrame(rows)
detailed_error_fidelity_df

## Baseline comparison table

In [ ]:
saved_comparison = pd.read_csv(CSV_DIR / "logreg_vs_ebm_vs_gaminet_comparison.csv")
mixed_comparison = saved_comparison[saved_comparison["variant"] == "mixed_300k_plus_vae"].copy()
display(mixed_comparison)

plot_df = mixed_comparison.melt(
    id_vars=["teacher", "surrogate"],
    value_vars=["overall_fidelity_accuracy", "error_fidelity_accuracy", "fp_fidelity_accuracy", "fn_fidelity_accuracy"],
    var_name="metric",
    value_name="score",
)

for metric in ["overall_fidelity_accuracy", "error_fidelity_accuracy", "fp_fidelity_accuracy", "fn_fidelity_accuracy"]:
    metric_df = plot_df[plot_df["metric"] == metric].pivot(index="teacher", columns="surrogate", values="score")
    metric_df = metric_df[["logistic_regression", "ebm", "gaminet"]]
    ax = metric_df.plot(kind="bar", figsize=(10, 4), ylim=(0, 1), color=["#607d8b", "#2f7d32", "#c62828"])
    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("Teacher")
    ax.set_ylabel("Score")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="Surrogate", loc="lower right")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## Save notebook-computed tables

In [ ]:
overall_fidelity_df.to_csv(CSV_DIR / "notebook_overall_fidelity_300k_vae.csv", index=False)
error_fidelity_df.to_csv(CSV_DIR / "notebook_error_fidelity_300k_vae.csv", index=False)
detailed_error_fidelity_df.to_csv(CSV_DIR / "notebook_detailed_error_fidelity_300k_vae.csv", index=False)
pd.DataFrame({"saved": [
    str(CSV_DIR / "notebook_overall_fidelity_300k_vae.csv"),
    str(CSV_DIR / "notebook_error_fidelity_300k_vae.csv"),
    str(CSV_DIR / "notebook_detailed_error_fidelity_300k_vae.csv"),
]})

## Feature importance plots

In [ ]:
feature_importance_sources = {
    ("random_forest", "EBM"): CSV_DIR / "ebm_rf_feature_importance.csv",
    ("random_forest", "GAMI-Net"): CSV_DIR / "gaminet_rf_feature_importance.csv",
    ("deep_neural_net", "EBM"): CSV_DIR / "ebm_dnn_feature_importance.csv",
    ("deep_neural_net", "GAMI-Net"): CSV_DIR / "gaminet_dnn_feature_importance.csv",
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, ((teacher, surrogate), path) in zip(axes.ravel(), feature_importance_sources.items()):
    importance_df = pd.read_csv(path).head(10).iloc[::-1]
    color = "#006eff"
    ax.barh(importance_df["feature"], importance_df["importance"], color=color, alpha=0.85)
    ax.set_title(f"{teacher.replace('_', ' ').title()} - {surrogate}")
    ax.set_xlabel("Importance")
    ax.grid(axis="x", alpha=0.25)
fig.suptitle("Generated Feature Importance Plots for 300k+VAE Surrogates", fontsize=16)
fig.tight_layout()
plt.show()

## Main effect term plots

In [ ]:
selected_features = pd.read_csv(CSV_DIR / "selected_common_top5_features.csv")

def display_feature_name(term_name):
    term_name = str(term_name)
    return term_name.split("__", 1)[1] if "__" in term_name else term_name


def find_ebm_main_term(model, feature):
    for idx, (term_name, term_features) in enumerate(zip(model.term_names_, model.term_features_)):
        if len(term_features) == 1 and display_feature_name(term_name) == feature:
            return idx, term_name
    raise KeyError(f"EBM main-effect term not found: {feature}")


def find_gaminet_main_term(global_dict, feature):
    for term_name, payload in global_dict.items():
        if payload.get("type") == "continuous" and display_feature_name(term_name) == feature:
            return term_name, payload
    raise KeyError(f"GAMI-Net main-effect term not found: {feature}")


def ebm_axis(names, score_count):
    x = np.asarray(names)
    if x.size == score_count + 1:
        return x[:-1]
    if x.size == score_count:
        return x
    return np.arange(score_count)


def plot_main_effect(ax, x, y, feature, surrogate, color):
    x = np.asarray(x)
    y = np.asarray(y, dtype=float)
    try:
        x_numeric = np.asarray(x, dtype=float)
        is_numeric = np.isfinite(x_numeric).all()
    except Exception:
        is_numeric = False
    if is_numeric and x_numeric.size >= 8:
        order = np.argsort(x_numeric)
        ax.plot(x_numeric[order], y[order], color=color, linewidth=2.1)
        ax.set_xlabel(feature)
    else:
        labels = [str(value) for value in x]
        if len(labels) != len(y):
            labels = [str(i) for i in range(len(y))]
        pos = np.arange(len(y))
        ax.bar(pos, y, color=color, alpha=0.82)
        if len(pos) <= 12:
            ax.set_xticks(pos)
            ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.set_xlabel(feature)
    ax.axhline(0, color="#222222", linewidth=0.8, alpha=0.55)
    ax.set_ylabel("Contribution")
    ax.set_title(surrogate)
    ax.grid(alpha=0.22)


for teacher in TEACHERS:
    teacher_features = selected_features[selected_features["teacher"] == teacher]["feature"].tolist()
    ebm_model = models[teacher]["ebm"]
    ebm_global = ebm_model.explain_global()
    gaminet_global = models[teacher]["gaminet"].global_explain(save_dict=False)

    fig, axes = plt.subplots(len(teacher_features), 2, figsize=(15, 3.7 * len(teacher_features)))
    if len(teacher_features) == 1:
        axes = np.asarray([axes])

    for row_idx, feature in enumerate(teacher_features):
        ebm_idx, ebm_term = find_ebm_main_term(ebm_model, feature)
        ebm_payload = ebm_global.data(ebm_idx)
        ebm_scores = np.asarray(ebm_payload["scores"], dtype=float)
        ebm_x = ebm_axis(ebm_payload.get("names", []), ebm_scores.size)
        plot_main_effect(axes[row_idx, 0], ebm_x, ebm_scores, feature, f"EBM main effect: {feature}", "#006eff")

        gaminet_term, gaminet_payload = find_gaminet_main_term(gaminet_global, feature)
        gaminet_x = np.asarray(gaminet_payload["inputs"])
        gaminet_y = np.asarray(gaminet_payload["outputs"], dtype=float)
        plot_main_effect(axes[row_idx, 1], gaminet_x, gaminet_y, feature, f"GAMI-Net main effect: {feature}", "#006eff")

    fig.suptitle(f"Generated Main-Effect Term Plots - {teacher.replace('_', ' ').title()}", fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()